In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------
import re
import datetime
import pandas as pd
from selenium import webdriver
import requests
from bs4 import BeautifulSoup
import time
from time import sleep
import os
from bs4 import BeautifulSoup
from selenium import webdriver
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# %%

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'IN SEBI' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process


if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running IN SEBI Web Scraping Tool v.2.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

# %%

In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def parse_page(html):
    soup = BeautifulSoup(html, "html.parser")

    totalpage = soup.select_one("input[name='totalpage']")
    nextvalue = soup.select_one("input[name='nextValue']")
    nextdel = soup.select_one("input[name='nextDel']")

    totalpage = int(totalpage["value"]) if totalpage and totalpage.has_attr("value") else None
    nextvalue = int(nextvalue["value"]) if nextvalue and nextvalue.has_attr("value") else None
    nextdel = nextdel["value"] if nextdel and nextdel.has_attr("value") else None

    entities = []
    for block in soup.select(".fixed-table-body"):
        entity = {}
        for card in block.select(".card-view"):
            label_el = card.select_one(".title")
            value_el = card.select_one(".value")
            if not label_el or not value_el:
                continue
            label = label_el.get_text(strip=True)
            value = value_el.get_text(strip=True)
            entity[label] = value
        if entity:
            entities.append(entity)

    return totalpage, nextvalue, nextdel, entities


def parse_page_reg_14(html):
    soup = BeautifulSoup(html, "html.parser")

    totalpage = soup.select_one("input[name='totalpage']")
    nextvalue = soup.select_one("input[name='nextValue']")
    nextdel = soup.select_one("input[name='nextDel']")

    totalpage = int(totalpage["value"]) if totalpage and totalpage.has_attr("value") else None
    nextvalue = int(nextvalue["value"]) if nextvalue and nextvalue.has_attr("value") else None
    nextdel = nextdel["value"] if nextdel and nextdel.has_attr("value") else None

    entities = []
    for row in soup.select("tbody > tr"):
        tds = row.find_all("td")
        if len(tds) < 6:
            continue

        name = tds[1].get_text(strip=True)
        reg_no = tds[2].get_text(strip=True)
        validity = tds[3].get_text(strip=True)

        address, email, contact_person, corr_address = parse_address_cell(tds[5])

        entity = {
            "Name": name,
            "Registration No": reg_no,
            "Validity": validity,
            "Address": address,
            "Email": email,
            "Contact Person": contact_person,
            "Correspondence Address": corr_address,
        }
        entities.append(entity)

    return totalpage, nextvalue, nextdel, entities


def fetch_page(next_value, URL, headers_info, payload_info):
    data = dict(payload_info)
    data["doDirect"] = str(next_value-1)
    resp = requests.post(URL, headers=headers_info,  data=data, timeout=30,verify=False)
    resp.raise_for_status()
    return resp.text

def parse_address_cell(td):
    parts = [p.strip() for p in td.get_text("\n", strip=True).split("\n") if p.strip()]
    address = parts[0] if parts else ""
    email = ""
    contact_person = ""
    corr_address = ""

    for p in parts[1:]:
        low = p.lower()
        if low.startswith("email:"):
            email = p.split(":", 1)[1].strip()
        elif low.startswith("contact person"):
            contact_person = p.split(":", 1)[1].strip()
        elif "correspondence address" in low:
            corr_address = p.split(":", 1)[1].strip()

    return address, email, contact_person, corr_address


def extract_zip_code(text):
    if not text:
        return ""
    m = re.search(r"\b\d{6}\b", text)
    return m.group(0) if m else ""

def parse_validity_dates(validity_text):
    if not validity_text:
        return "", ""
    # check if any month name is present
    if not re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", validity_text, re.I):
        return "", ""
    parts = [p.strip() for p in validity_text.split("-")]
    register_date = parts[0] if parts else ""
    cancellation_date = parts[-1] if len(parts) > 1 else ""
    return register_date, cancellation_date



In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         'IN SEBI 1': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=5',
        'IN SEBI 2': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=7',
        'IN SEBI 3': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=6',
        'IN SEBI 4': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=29',
        'IN SEBI 5': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=13',
        'IN SEBI 6': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=8',
        # all expired .....
        'IN SEBI 7': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=9',
        'IN SEBI 8': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=16',
        'IN SEBI 9': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=27',
        'IN SEBI 10': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=18',
        'IN SEBI 11': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=19',
        'IN SEBI 12': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=47',
        'IN SEBI 13': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=25',
         'IN SEBI 14': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=20',
        # Differenct Table need to omprove new parse html 
        'IN SEBI 15': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=23',
        'IN SEBI 16': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=33',
        'IN SEBI 17': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=2',
        'IN SEBI 18': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=32',
        'IN SEBI 19': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=37',
        'IN SEBI 20': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=31',
        'IN SEBI 21': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=38',
        'IN SEBI 22': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=30',
        'IN SEBI 23': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=21',
        'IN SEBI 24': 'https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes&intmId=10',
        }



Typology={

'IN SEBI 1':    'Banker to an Issue',
'IN SEBI 2':	'Credit Rating Agency - CRA',
'IN SEBI 3':	'Debentures Trustee',
'IN SEBI 4':	'FPIs / Deemed FPIs (Erstwhile FIIs/QFIs)',
'IN SEBI 5':	'Investment Adviser',
'IN SEBI 6':	'KYC (Know Your Client) Registration Agency registered with SEBI',
'IN SEBI 7':	'Merchant Bankers',
'IN SEBI 8':	'Registered Alternative Investment Funds',
'IN SEBI 9':	'Registered Custodians',
'IN SEBI 10':	'Registered Depository Participants - CDSL',
'IN SEBI 11':	'Registered Depository Participants - NSDL',
'IN SEBI 12':	'Registered ESG Rating Providers',
'IN SEBI 13':	'Registered Foreign Venture Capital Investors',
'IN SEBI 14':	'Registered Infrastructure Investment Trusts',
'IN SEBI 15':	'Registered Mutual Funds',
'IN SEBI 16':	'Registered Portfolio Managers',
'IN SEBI 17':	'Registered Stock Brokers in Commodity Derivative Segment',
'IN SEBI 18':	'Registered Stock Brokers in Currency Derivative Segment',
'IN SEBI 19':	'Registered Stock Brokers in Debt Segment',
'IN SEBI 20':	'Registered Stock Brokers in Equity Derivative Segment',
'IN SEBI 21':	'Registered Stock Brokers in Interest Rate Derivative Segment',
'IN SEBI 22':	'Registered Stock Brokers in equity segment',
'IN SEBI 23':	'Registered Venture Capital Funds',
'IN SEBI 24':	'Registrars to an issue and share Transfer Agents',


        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

	  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')


URL = "https://www.sebi.gov.in/sebiweb/ajax/other/getintmfpiinfo.jsp"

HEADERS = {
    "accept": "*/*",
    "content-type": "application/x-www-form-urlencoded",
    "origin": "https://www.sebi.gov.in",
    "referer": "https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doRecognisedFpi=yes",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36 Edg/144.0.0.0",
    "x-ps-ext": "2024",
}


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    BASE_DATA = {
        "next":'n',
        "intmId": regdict[reg].split('=')[-1],
        "language": "2",
        "doDirect":"0"
    }

    # first page
    html = fetch_page(1,URL,HEADERS,BASE_DATA)
    totalpage, nextvalue, nextdel, entities = parse_page(html)
    print("totalpage:", totalpage, "nextvalue:", nextvalue, "nextdel:", nextdel)
    #print("entities on page 1:", len(entities))

    # loop all pages
    all_entities = []
    if totalpage:
        for nv in range(1, totalpage + 1):
            html = fetch_page(nv,URL,HEADERS,BASE_DATA)
            if reg != regulatorName+' 14':
                _, _, _, entities = parse_page(html)
            else:
                _, _, _, entities = parse_page_reg_14(html)
            all_entities.extend(entities)

    print("total entities:", len(all_entities))


    for entity in all_entities:
        name_ = entity.get('Name') or entity.get('Trade Name') or''
        address_ = entity.get("Address") or entity.get("Correspondence Address") or ""

        email_ = entity.get('E-mail') or entity.get('Correspondence E-mail') or ''
        tele_  = entity.get('Telephone') or entity.get('Correspondence Telephone') or ''
        validate_date = entity.get('Validity') or ''
        fax_ = entity.get('Fax No.') or ''
        register_number = entity.get('Registration No.') or ''
        country_name_ = entity.get('Country Name') or ''
        model_ = entity.get('Model') or ''
        exchange_name = entity.get('Exchange Name') or ''
        zip_code = extract_zip_code(address_)
        register_date, cancellation_date = parse_validity_dates(validate_date)


        sqldict['Name'].append(name_)
        sqldict['InternalID_1'].append(register_number)
        sqldict['InternalID_1_type'].append('Registration No.')
        sqldict['Address_1'].append(address_)
        sqldict['Zip'].append(zip_code)
        sqldict['Cntry'].append(country_name_)
        sqldict['Typology'].append(model_)
        sqldict['Phone'].append(tele_)
        sqldict['Fax'].append(fax_)
        sqldict['Name - Mother Company'].append(exchange_name)
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegulationDate'].append(register_date)
        sqldict['CancellationDate'].append(cancellation_date)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(Typology[reg])

        sqldict = bourange_same_length_array(sqldict)


    


[INFO] : Working 1/24 _(IN SEBI 1)_ 
totalpage: 3 nextvalue: 1 nextdel: 25
total entities: 60
[INFO] : Working 2/24 _(IN SEBI 2)_ 
totalpage: 1 nextvalue: 1 nextdel: 8
total entities: 8
[INFO] : Working 3/24 _(IN SEBI 3)_ 
totalpage: 2 nextvalue: 1 nextdel: 25
total entities: 26
[INFO] : Working 4/24 _(IN SEBI 4)_ 
totalpage: 474 nextvalue: 1 nextdel: 25
total entities: 11835
[INFO] : Working 5/24 _(IN SEBI 5)_ 
totalpage: 40 nextvalue: 1 nextdel: 25
total entities: 980
[INFO] : Working 6/24 _(IN SEBI 6)_ 
totalpage: 1 nextvalue: 1 nextdel: 6
total entities: 6
[INFO] : Working 7/24 _(IN SEBI 7)_ 
totalpage: 10 nextvalue: 1 nextdel: 25
total entities: 238
[INFO] : Working 8/24 _(IN SEBI 8)_ 
totalpage: 71 nextvalue: 1 nextdel: 25
total entities: 1754
[INFO] : Working 9/24 _(IN SEBI 9)_ 
totalpage: 1 nextvalue: 1 nextdel: 17
total entities: 17
[INFO] : Working 10/24 _(IN SEBI 10)_ 
totalpage: 29 nextvalue: 1 nextdel: 25
total entities: 718
[INFO] : Working 11/24 _(IN SEBI 11)_ 
totalpage

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df[df['Name']!='']
s = df["CancellationDate"].astype(str).str.strip()
is_perpetual = s.str.casefold().eq("perpetual")
parsed = pd.to_datetime(s, errors="coerce")
today = pd.Timestamp("today").normalize()

df_clean = df[is_perpetual | (parsed >= today) | parsed.isna()]

# df_expire = df[~is_perpetual & (parsed < today)]
# df_expire = df_expire[df_expire["CancellationDate"].str.casefold().ne("perpetual")]
# df  = df.drop_duplicates()

df_clean = df_clean.applymap(lambda x: x.encode('unicode_escape').
                 decode('utf-8') if isinstance(x, str) else x)

df_clean.to_excel(filename, index=False)

driver.quit()

sleep(3)


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_18860\2762227442.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.encode('unicode_escape').


In [ ]:
df = df[df['Name']!='']
s = df["CancellationDate"].astype(str).str.strip()
is_perpetual = s.str.casefold().eq("perpetual")
parsed = pd.to_datetime(s, errors="coerce")
today = pd.Timestamp("today").normalize()

df_clean = df[is_perpetual | (parsed >= today) | parsed.isna()]

df_expire = df[~is_perpetual & (parsed < today)]
df_expire = df_expire[df_expire["CancellationDate"].str.casefold().ne("perpetual")]


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_18860\1797773315.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="coerce")


In [44]:
df_clean.to_excel('date_expire.xlsx')